<a href="https://colab.research.google.com/github/ajay-navodayan/Ajay_NER-Project-of-Health-record-Deep-Learning-/blob/main/AjayDL_Projrct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

In [ ]:
!pip install datasets==2.18.0

In [ ]:
dataset = load_dataset("ncbi_disease")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1461: FutureWarning: The repository for ncbi_disease contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/ncbi_disease
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/5433 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/924 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/941 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 924
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 941
    })
})

In [ ]:
dataset["train"][0]

{'id': '0',
 'tokens': ['Identification',
  'of',
  'APC2',
  ',',
  'a',
  'homologue',
  'of',
  'the',
  'adenomatous',
  'polyposis',
  'coli',
  'tumour',
  'suppressor',
  '.'],
 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0]}

In [ ]:
model_name = "dmis-lab/biobert-base-cased-v1.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=3)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored 

In [ ]:
def tokenize_and_align_labels(example):

    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    word_ids = tokenized_inputs.word_ids()

    labels = []
    previous_word_idx = None

    for word_idx in word_ids:

        if word_idx is None:
            labels.append(-100)

        elif word_idx != previous_word_idx:
            labels.append(example["ner_tags"][word_idx])

        else:
            labels.append(example["ner_tags"][word_idx])

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=456710c062dbc4efab475a7b0be9c5d0ef5325e06bf3d46a2f643fb7385b47d5
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
from seqeval.metrics import f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    label_list = ["O", "B-Disease", "I-Disease"]

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="ner_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3
)

In [ ]:
# 1. Import Trainer
from transformers import Trainer

# 2. Define compute_metrics (you already added)

# 3. Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

trainer.evaluate()

Step,Training Loss
500,0.125602
1000,0.040282
1500,0.026129
2000,0.015155


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.07672898471355438,
 'eval_precision': 0.8399532710280374,
 'eval_recall': 0.8768292682926829,
 'eval_f1': 0.8579952267303103,
 'eval_runtime': 4.0226,
 'eval_samples_per_second': 229.704,
 'eval_steps_per_second': 28.837,
 'epoch': 3.0}

In [ ]:
model.save_pretrained("biobert_disease_model")
tokenizer.save_pretrained("biobert_disease_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('biobert_disease_model/tokenizer_config.json',
 'biobert_disease_model/tokenizer.json')

In [ ]:
ner = pipeline(
    "ner",
    model="biobert_disease_model",
    tokenizer="biobert_disease_model",
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
text = "The patient was diagnosed with diabetes and lung cancer."

results = ner(text)

for r in results:
    print(r)

{'entity_group': 'LABEL_0', 'score': np.float32(0.9998353), 'word': 'The patient was diagnosed with', 'start': 0, 'end': 30}
{'entity_group': 'LABEL_1', 'score': np.float32(0.9879981), 'word': 'diabetes', 'start': 31, 'end': 39}
{'entity_group': 'LABEL_0', 'score': np.float32(0.9988601), 'word': 'and', 'start': 40, 'end': 43}
{'entity_group': 'LABEL_1', 'score': np.float32(0.9611082), 'word': 'lung', 'start': 44, 'end': 48}
{'entity_group': 'LABEL_2', 'score': np.float32(0.99706453), 'word': 'cancer', 'start': 49, 'end': 55}
{'entity_group': 'LABEL_0', 'score': np.float32(0.9998171), 'word': '.', 'start': 55, 'end': 56}


In [ ]:
text = "Cancer patient was diagnosed and now he is very well but feeling suffers from diabetes."

results = ner(text)

for r in results:
    print(r)

{'entity_group': 'LABEL_1', 'score': np.float32(0.9851125), 'word': 'Cancer', 'start': 0, 'end': 6}
{'entity_group': 'LABEL_0', 'score': np.float32(0.9997395), 'word': 'patient was diagnosed and now he is very well but feeling suffers from', 'start': 7, 'end': 77}
{'entity_group': 'LABEL_1', 'score': np.float32(0.91121835), 'word': 'diabetes', 'start': 78, 'end': 86}
{'entity_group': 'LABEL_0', 'score': np.float32(0.9997743), 'word': '.', 'start': 86, 'end': 87}
